# Part 2 — Constant-area ellipse optimization (sweet spot)

Sweeps the aspect ratio at approximately constant total area and computes the ratio of effective (sweet-spot) area to total area, identifying the best symmetric ellipse (Figure 6b, 6c in the manuscript).

In [ ]:
import numpy as np
from numba import njit, prange
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
import multiprocessing as mp

# ------------------------------
# Boundary Generators
# ------------------------------
def make_rect_edge(nx, ny):
    edges = np.zeros((ny, nx), dtype=np.bool_)
    edges[0, :] = True
    edges[-1, :] = True
    edges[:, 0] = True
    edges[:, -1] = True
    return edges

def make_ellipse_edge(nx, ny, aspect_ratio=1.0):
    """
    aspect_ratio: major-axis / minor-axis ratio (>= 1.0)
    aspect_ratio = 1.0: circle
    aspect_ratio > 1.0: ellipse (elongated along x)
    
    Total area is kept approximately constant by adjusting radii:
    Area = π * rx * ry = constant
    """
    edges = np.zeros((ny, nx), dtype=np.bool_)
    cx, cy = (nx - 1) / 2, (ny - 1) / 2
    
    # Keep area constant: rx * ry = constant
    # For aspect_ratio = 1: rx = ry = min(cx, cy)
    # For aspect_ratio > 1: rx increases, ry decreases proportionally
    base_radius = min(cx, cy)
    rx = base_radius * np.sqrt(aspect_ratio)
    ry = base_radius / np.sqrt(aspect_ratio)
    
    for y in range(ny):
        for x in range(nx):
            dx = (x - cx) / rx if rx > 0 else 0
            dy = (y - cy) / ry if ry > 0 else 0
            if dx * dx + dy * dy >= 1.0:
                edges[y, x] = True
    return edges

def calculate_grid_size_for_constant_area(base_nx, base_ny, aspect_ratio, target_area):
    """
    Use fixed grid size to avoid discretization issues.
    The ellipse itself maintains constant area through rx * ry = constant.
    By keeping the grid fixed, we ensure fair comparison.
    """
    # Keep grid size constant - the ellipse area is maintained by the radius formula
    return base_nx, base_ny

# ------------------------------
# Relaxation Solver
# ------------------------------
@njit(cache=True, fastmath=True)
def _relax(height, edges, dot_y, dot_x, dot_height, max_iter=50000, tol=1e-8):
    ny, nx = height.shape
    for _ in range(max_iter):
        max_diff = 0.0
        for y in range(1, ny - 1):
            for x in range(1, nx - 1):
                if x == dot_x and y == dot_y:
                    height[y, x] = dot_height
                    continue
                if edges[y, x]:
                    height[y, x] = 0.0
                    continue
                new_val = 0.25 * (
                    height[y + 1, x] +
                    height[y - 1, x] +
                    height[y, x + 1] +
                    height[y, x - 1]
                )
                diff = abs(new_val - height[y, x])
                if diff > max_diff:
                    max_diff = diff
                height[y, x] = new_val
        if max_diff < tol:
            break
    return height

def surface(nx, ny, dot_pos, dot_height, edges):
    height = np.zeros((ny, nx), dtype=np.float64)
    dot_y, dot_x = dot_pos
    return _relax(height, edges, dot_y, dot_x, dot_height)

@njit(parallel=True, cache=True, fastmath=True)
def _calculate_forces_parallel(nx, ny, dot_height, k, edges):
    """Parallelized force calculation using numba."""
    forces = np.zeros((ny, nx), dtype=np.float64)
    
    # Calculate forces for ALL interior points
    for y in prange(ny):
        for x in range(nx):
            if edges[y, x]:
                continue
            
            # Calculate surface for this point
            height_map = np.zeros((ny, nx), dtype=np.float64)
            height_map = _relax(height_map, edges, y, x, dot_height, max_iter=50000, tol=1e-8)
            
            # Count valid neighbors and sum their heights
            neighbors_sum = 0.0
            neighbor_count = 0
            
            if y > 0:
                neighbors_sum += height_map[y-1, x]
                neighbor_count += 1
            if y < ny-1:
                neighbors_sum += height_map[y+1, x]
                neighbor_count += 1
            if x > 0:
                neighbors_sum += height_map[y, x-1]
                neighbor_count += 1
            if x < nx-1:
                neighbors_sum += height_map[y, x+1]
                neighbor_count += 1
            
            # Force calculation: expected height at center minus actual average
            forces[y, x] = k * (neighbor_count * dot_height - neighbors_sum)
    
    return forces

# ------------------------------
# Force Landscape
# ------------------------------
def force_landscape(nx, ny, dot_height=1.0, k=1.0, shape='rect', aspect_ratio=1.0):
    if shape == 'rect':
        edges = make_rect_edge(nx, ny)
    elif shape == 'ellipse':
        edges = make_ellipse_edge(nx, ny, aspect_ratio)
    else:
        raise ValueError("shape must be 'rect' or 'ellipse'")
    
    # Use parallelized version
    forces = _calculate_forces_parallel(nx, ny, dot_height, k, edges)
    
    return forces, edges

# ------------------------------
# Find group edges
# ------------------------------
def group_edges(force_map, edges, alpha=1.2, method='relative'):
    """
    Find the effective hitting area based on force distribution.
    
    Parameters:
    - force_map: 2D array of forces
    - edges: boolean mask of boundary edges
    - alpha: threshold parameter
    - method: 'relative' (alpha * min_force) or 'percentile' (top alpha*100%)
    """
    # Get interior points (not edges)
    interior_mask = ~edges
    interior_forces = force_map[interior_mask]
    
    if len(interior_forces) == 0:
        return np.zeros_like(force_map, dtype=bool), np.zeros_like(force_map, dtype=bool)
    
    if method == 'percentile':
        # Use percentile-based approach: select points with force <= alpha-th percentile
        # alpha=1.2 means top 120% (but capped at 100%), so we use min(alpha*50, 100) percentile
        percentile_value = np.percentile(interior_forces, min(alpha * 50, 100))
        group_mask = (force_map <= percentile_value) & interior_mask
    else:
        # Original relative method
        min_force = np.min(interior_forces)
        group_mask = (force_map <= alpha * min_force) & interior_mask
    
    ny, nx = force_map.shape
    edge_mask = np.zeros_like(force_map, dtype=bool)
    
    for y in range(ny):
        for x in range(nx):
            if not group_mask[y, x]:
                continue
            neighbors = [
                group_mask[y-1, x] if y > 0 else False,
                group_mask[y+1, x] if y < ny-1 else False,
                group_mask[y, x-1] if x > 0 else False,
                group_mask[y, x+1] if x < nx-1 else False
            ]
            if not all(neighbors):
                edge_mask[y, x] = True
    return edge_mask, group_mask

# ------------------------------
# Optimize Racket Shape
# ------------------------------
def calculate_efficiency(base_nx, base_ny, aspect_ratio, alpha=1.2, dot_height=1.0, k=1.0, method='relative', use_adaptive_grid=True, target_area=None):
    """
    Compute the effective-area-per-unit-area ratio for a given aspect_ratio
    
    Parameters:
    - use_adaptive_grid: If True, adjust grid size to maintain constant total area
    - target_area: Target interior area (number of dots)
    """
    if use_adaptive_grid and target_area is not None:
        nx, ny = calculate_grid_size_for_constant_area(base_nx, base_ny, aspect_ratio, target_area)
    else:
        nx, ny = base_nx, base_ny
    
    forces, edges = force_landscape(nx, ny, dot_height=dot_height, k=k, 
                                     shape='ellipse', aspect_ratio=aspect_ratio)
    edge_group_mask, group_mask = group_edges(forces, edges, alpha, method=method)
    
    # total interior area (excluding boundary)
    area_total = np.sum(~edges)
    
    # effective hitting area (Rule 2 group)
    area_effective = np.sum(group_mask)
    
    # efficiency: effective area / total area
    efficiency = area_effective / area_total if area_total > 0 else 0
    
    # diagnostic info
    interior_forces = forces[~edges]
    min_force = np.min(interior_forces) if len(interior_forces) > 0 else 0
    max_force = np.max(interior_forces) if len(interior_forces) > 0 else 0
    mean_force = np.mean(interior_forces) if len(interior_forces) > 0 else 0
    threshold = alpha * min_force
    
    return efficiency, area_total, area_effective, min_force, max_force, mean_force, threshold, nx, ny

def find_optimal_racket(nx, ny, alpha=1.2, aspect_ratios=None, method='relative', use_adaptive_grid=True):
    """
    Test various aspect_ratio values to find the optimal racket head shape
    
    Parameters:
    - nx, ny: base grid size
    - alpha: threshold for the effective-area criterion
    - aspect_ratios: list of major/minor ratios to test
    - method: 'relative' or 'percentile'
    - use_adaptive_grid: If True, adjust grid size to keep total area constant
    
    Returns:
    - results: list of result dictionaries, one per aspect_ratio
    - optimal_ratio: the optimal aspect_ratio
    """
    if aspect_ratios is None:
        aspect_ratios = np.linspace(1.0, 1.1, 11)  # 1.0 to 1.1 in steps of 0.01
    
    # Calculate target area from aspect_ratio=1.0 (circle)
    if use_adaptive_grid:
        _, edges_reference = force_landscape(nx, ny, shape='ellipse', aspect_ratio=1.0)
        target_area = np.sum(~edges_reference)
        print(f"Target area (from circle): {target_area} dots")
    else:
        target_area = None
    
    results = []
    
    mode_str = "Adaptive Grid (Constant Area)" if use_adaptive_grid else "Fixed Grid"
    print(f"\n{'='*100}")
    print(f"Badminton Racket Optimization Analysis - {mode_str}")
    print(f"Base: nx={nx}, ny={ny}, alpha={alpha}, method={method}")
    print(f"{'='*100}")
    print(f"{'Ratio':<8} {'GridSize':<12} {'Total':<8} {'Effective':<10} {'Efficiency':<12} {'MinF':<8} {'MeanF':<8} {'Thresh':<8}")
    print(f"{'-'*100}")
    
    for ratio in aspect_ratios:
        efficiency, total_area, effective_area, min_force, max_force, mean_force, threshold, actual_nx, actual_ny = calculate_efficiency(
            nx, ny, ratio, alpha=alpha, method=method, 
            use_adaptive_grid=use_adaptive_grid, target_area=target_area
        )
        
        result = {
            'aspect_ratio': ratio,
            'grid_size': (actual_nx, actual_ny),
            'total_area': total_area,
            'effective_area': effective_area,
            'efficiency': efficiency,
            'min_force': min_force,
            'max_force': max_force,
            'mean_force': mean_force,
            'threshold': threshold
        }
        results.append(result)
        
        grid_str = f"{actual_nx}x{actual_ny}"
        print(f"{ratio:<8.3f} {grid_str:<12} {total_area:<8.0f} {effective_area:<10.0f} {efficiency:<12.4f} {min_force:<8.4f} {mean_force:<8.4f} {threshold:<8.4f}")
    
    # find the optimal aspect_ratio
    optimal_idx = np.argmax([r['efficiency'] for r in results])
    optimal_ratio = results[optimal_idx]['aspect_ratio']
    
    print(f"{'-'*100}")
    print(f"Optimal Aspect Ratio: {optimal_ratio:.3f}")
    print(f"Maximum Efficiency: {results[optimal_idx]['efficiency']:.4f}")
    print(f"{'='*100}\n")
    
    return results, optimal_ratio

def visualize_optimization_results(results):
    """
    Visualize the optimization results as plots
    """
    aspect_ratios = [r['aspect_ratio'] for r in results]
    efficiencies = [r['efficiency'] for r in results]
    total_areas = [r['total_area'] for r in results]
    effective_areas = [r['effective_area'] for r in results]
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    # 1. Efficiency vs Aspect Ratio
    axes[0, 0].plot(aspect_ratios, efficiencies, 'b-o', linewidth=2, markersize=5)
    optimal_idx = np.argmax(efficiencies)
    axes[0, 0].plot(aspect_ratios[optimal_idx], efficiencies[optimal_idx], 
                     'r*', markersize=15, label='Optimal')
    axes[0, 0].set_xlabel('Aspect Ratio (Major/Minor)')
    axes[0, 0].set_ylabel('Efficiency (Effective/Total)')
    axes[0, 0].set_title('Efficiency Ratio per Unit Area')
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].legend()
    
    # 2. Total Area vs Aspect Ratio
    axes[0, 1].plot(aspect_ratios, total_areas, 'g-o', linewidth=2, markersize=5)
    axes[0, 1].set_xlabel('Aspect Ratio (Major/Minor)')
    axes[0, 1].set_ylabel('Total Area')
    axes[0, 1].set_title('Total Area Change')
    axes[0, 1].grid(True, alpha=0.3)
    
    # 3. Effective Area vs Aspect Ratio
    axes[1, 0].plot(aspect_ratios, effective_areas, 'm-o', linewidth=2, markersize=5)
    axes[1, 0].set_xlabel('Aspect Ratio (Major/Minor)')
    axes[1, 0].set_ylabel('Effective Area')
    axes[1, 0].set_title('Effective Area Change')
    axes[1, 0].grid(True, alpha=0.3)
    
    # 4. Both areas comparison
    axes[1, 1].plot(aspect_ratios, total_areas, 'g-o', linewidth=2, 
                    markersize=5, label='Total Area')
    axes[1, 1].plot(aspect_ratios, effective_areas, 'm-o', linewidth=2, 
                    markersize=5, label='Effective Area')
    axes[1, 1].set_xlabel('Aspect Ratio (Major/Minor)')
    axes[1, 1].set_ylabel('Area')
    axes[1, 1].set_title('Area Comparison')
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].legend()
    
    plt.tight_layout()
    plt.show()

# ------------------------------
# Main Function
# ------------------------------
def main():
    """
    Main entry point
    """
    # parameter setup
    nx, ny = 60, 60  # large fixed grid so the ellipse always fits
    alpha = 1.2
    method = 'relative'  # 'relative' or 'percentile'
    use_adaptive_grid = True  # keep the area constant
    
    # 1. run the optimization
    aspect_ratios = np.linspace(1.0, 2.0, 20)  # 1.0 to 2.0 in steps of ~0.05
    results, optimal_ratio = find_optimal_racket(nx, ny, alpha=alpha, 
                                                  aspect_ratios=aspect_ratios,
                                                  method=method,
                                                  use_adaptive_grid=use_adaptive_grid)
    
    # 2. visualize the optimization results
    visualize_optimization_results(results)
    
    # 3. 3D visualization for optimal ratio
    print("Visualizing 3D force distribution of optimal racket shape...")
    forces, edges = force_landscape(nx, ny, dot_height=1.0, k=1.0, 
                                     shape='ellipse', aspect_ratio=optimal_ratio)
    forces_plot = np.ma.array(forces, mask=edges)
    edge_group_mask, group_mask = group_edges(forces, edges, alpha)
    
    X, Y = np.meshgrid(np.arange(nx), np.arange(ny))
    z_min = np.min(forces_plot) - 0.1
    
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    ax.plot_surface(X, Y, forces_plot, cmap='inferno', alpha=0.8)
    ax.scatter(X[edge_group_mask], Y[edge_group_mask], forces[edge_group_mask],
               color='cyan', s=50, label='Group Edge')
    ax.scatter(X[edges], Y[edges], z_min, color='black', s=30, label='Boundary Edge (bottom)')
    ax.scatter(X[edge_group_mask], Y[edge_group_mask], z_min, color='cyan', s=30, 
               label='Group Edge (bottom)')
    
    ax.set_title(f'Optimal Racket Shape (Aspect Ratio={optimal_ratio:.3f}, alpha={alpha})')
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Force')
    ax.legend()
    
    plt.show()

# ------------------------------
# Execute
# ------------------------------
if __name__ == "__main__":
    main()
